### To showcase that words and characters can be simply counted via normal function calling rather than trying llm to generate the count on it's own

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import display
load_dotenv()
GROQ_BASE_URL = os.getenv("GROQ_BASE_URL")
GROQ_API_KEY = os.getenv("GROQAI_API_KEY")
groq = OpenAI(base_url = GROQ_BASE_URL, api_key = GROQ_API_KEY)
model = os.getenv("GROQ_MODEL")

In [2]:
count_words_schema = {
    "name": "count_words",
    "description": "Count the number of words in the given user prompt",
    "parameters": {
        "type": "object",
        "properties": {
            "sentence": {
                "type": "string",
                "description": "The sentence that we have to count the words present",
            },
        },
        "required": ["sentence"],
        "additionalProperties": False
    }
}
character_occurs_schema = {
    "name":"count_char",
    "description": "Get the occurence of a character in the given sentence",
    "parameters":{
        "type":"object",
        "properties":{
            "sentence": {
                "type":"string",
                "description":"The sentence we have to count the characters for"
            },
            "character":{
                "type":"string",
                "description":"The character that we have to count the occurences in the given sentence"
            }
        },
        "required":["sentence", "character"],
        "additionalProperties":False
    }
}

system_message = "You are a helpful assistant. If user asks help regarding the counting words and occurence of a character, you should use tools and not generate randomly"

tools = [{"type": "function", "function": count_words_schema}, {"type":"function", "function":character_occurs_schema}]

In [3]:
def count_words(sentence: str) -> int:
    print("Count words called")
    return f"Total words in sentence: {len(sentence.split())}"


def count_char(sentence: str, character: str) -> int:
    print("Count Char called")
    return f"{character} present {sentence.count(character)} times in {sentence}"

In [4]:
AVAILABLE_TOOLS = {
    "count_words": count_words,
    "count_char": count_char
}

def handle_tool_calls(message):
    responses = []
    
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        # will get the arguments - select the function and call with the arguments
        print(tool_call)
        result = AVAILABLE_TOOLS[tool_call.function.name](**arguments)
        
        responses.append({
            "role": "tool",
            "content": result, 
            "tool_call_id": tool_call.id
        })
        
    return responses

In [5]:
def chat(message, history):
    #Formating history that gradio provides us directly
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message #llm specifies what tool to call with parameters
        responses = handle_tool_calls(message)
        messages.append(message)  # appending that tool needing message
        messages.extend(responses) # appending the responses from the tool
        response = groq.chat.completions.create(model=model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ChatCompletionMessageFunctionToolCall(id='fc_7dd30757-ce8e-4d40-8f6c-8da9085ace92', function=Function(arguments='{"sentence":"Help me in finding the number of words in this sentence"}', name='count_words'), type='function')
Count words called
ChatCompletionMessageFunctionToolCall(id='fc_b2cc76bf-c5e3-4224-819a-c3e7a02db6a1', function=Function(arguments='{"character":"a","sentence":"How much time does a occurs in this sentence?"}', name='count_char'), type='function')
Count Char called
ChatCompletionMessageFunctionToolCall(id='fc_2a9d0ff3-3b3d-4f1a-9c37-ded2ecfe5cc3', function=Function(arguments='{"character":"a","sentence":"a a a a a"}', name='count_char'), type='function')
Count Char called
